# Exercise 5.2 — Hyperparameters: choosing the lasso and ridge penalty (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-02-regularisation.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Some settings of a model are not learnt from the data: we choose them before
fitting. They are called **hyperparameters**. In this exercise the
hyperparameters are those of **regularisation**: a linear regression that pays
a price for large coefficients, so that it fits the training rows a little less
closely in exchange for a simpler model. How large the price is, $\lambda$,
must be chosen, and we choose it by trying several values and keeping the one
that predicts best on rows the model has not seen. The page
[Lasso and ridge in a picture](day5-01b-lasso-ridge-picture.qmd) explains what
the penalty does.

**Theory:** hyperparameters; grid search; why candidates are scored on a
validation set and not on the training rows; `tune_grid()` and
`GridSearchCV()`; refitting the chosen model and checking it once on the test
rows.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, PredefinedSplit, ValidationCurveDisplay
from sklearn.metrics import root_mean_squared_error

The same 750 people and the same split as Exercise 5.1, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv). The outcome is glucose
again, but this time the other seven measurements are the predictors. In
`pressure`, `triceps` and `insulin` a zero means "not measured", so we turn the
zeros into missing values and let the pipeline fill them in.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv").drop(columns="diabetes")   # glucose is the outcome
for col in ["pressure", "triceps", "insulin"]:
    diabetes[col] = diabetes[col].replace(0, np.nan)                         # 0 means "not measured"

predictors = ["pregnant", "pressure", "triceps", "insulin", "mass", "pedigree", "age"]

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

------------------------------------------------------------------------

### Part 1 — A model with room to overfit

The pipeline fills in the missing values with the median (Day 2), adds every
square and every product of two predictors with
`PolynomialFeatures(degree=2)`, and standardises the result. Seven predictors
become 35 columns: plenty of room to fit noise.

The lasso adds a penalty $\lambda \times \sum_j |\beta_j|$ to the fitting (see
[Lasso and ridge in a picture](day5-01b-lasso-ridge-picture.qmd)). $\lambda$ is
a **hyperparameter**: a setting we choose before fitting, which the fitting
does not choose for us. The degree of the polynomial in Exercise 5.1 was
another. In scikit-learn the lasso is `Lasso()`, and $\lambda$ is its `alpha`.
(`max_iter` lets the fitting run longer than the default, because the squares
and products are correlated and take longer to fit.)

In [ ]:
lasso_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),                # fill the missing values
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),  # squares and products
    ("scale", StandardScaler()),                                 # one scale for every column
    ("model", Lasso(max_iter=100000)),
])

The hyperparameter of a step in a pipeline is set with `set_params()`, using
the step's name, two underscores and the setting's name: `model__alpha`.

------------------------------------------------------------------------

### Part 2 — A grid search, scored on the training rows

A **grid search** tries a list of values, fits one model for each, and keeps
the one with the lowest error. Try seven penalties, each about three times the
one before. Fit each model on the training rows and, for now, measure its RMSE
on the **same** rows:

> **Stuck?**
>
> For now, the model is scored on the rows it was fitted on.

> **Worked answer**

In [ ]:
penalties = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
rows = []

for p in penalties:
    lasso_pipe.set_params(model__alpha=p)                  # fill in the hyperparameter
    lasso_pipe.fit(train[predictors], train["glucose"])

    train_rmse = root_mean_squared_error(train["glucose"], lasso_pipe.predict(train[predictors]))
    rows.append({"penalty": p, "train_rmse": train_rmse})

train_results = pd.DataFrame(rows)
train_results.round(2)

The RMSE only ever goes up: 26.83 at a penalty of 0.01, 30.78 at 10. Scored on
its own training rows, the smallest penalty always wins, and no penalty at all
would win outright.

**Question!** Does that mean the penalty is useless?

> **Learning points**
>
> - A grid search fits one model for each candidate value and keeps the candidate with the lowest error.
> - On the rows it was fitted on, a smaller penalty always looks better, because a freer model can follow those rows, noise included, more closely.

------------------------------------------------------------------------

### Part 3 — The same grid search, scored on a validation set

The training RMSE cannot tell us when the model starts fitting noise, because
the noise is in the rows it is measured on. We need rows the model has **not**
seen. The test rows are kept for one final check of the model we choose: if we
also used them to *make* the choice, we would pick whichever candidate
happened to do best on those 150 people, and its test RMSE would no longer be
an honest estimate (the Stretch of Exercise 5.1).

So we split the training rows once more. Every fourth training row becomes a
**validation** row: each candidate is fitted on the other rows and scored on
these.

In [ ]:
is_val = np.arange(len(train)) % 4 == 0          # every fourth training row
fitting    = train[~is_val]
validation = train[is_val]

print(len(fitting), len(validation))

Run the grid search again: fit on `fitting`, score on both sets.

> **Stuck?**
>
> Each candidate is fitted on the 450 `fitting` rows and scored on the 150 rows
> it has not seen.

> **Worked answer**

In [ ]:
rows = []

for p in penalties:
    lasso_pipe.set_params(model__alpha=p)
    lasso_pipe.fit(fitting[predictors], fitting["glucose"])

    fit_rmse = root_mean_squared_error(fitting["glucose"], lasso_pipe.predict(fitting[predictors]))
    val_rmse = root_mean_squared_error(validation["glucose"], lasso_pipe.predict(validation[predictors]))
    rows.append({"penalty": p, "fit_rmse": fit_rmse, "val_rmse": val_rmse})

val_results = pd.DataFrame(rows)
val_results.round(2)

In [ ]:
fig, ax = plt.subplots()
ax.plot(val_results["penalty"], val_results["fit_rmse"], marker="o", label="fit_rmse")
ax.plot(val_results["penalty"], val_results["val_rmse"], marker="o", label="val_rmse")
ax.set_xscale("log")
ax.set_xlabel("Penalty (log scale)")
ax.set_ylabel("RMSE (mg/dL)")
ax.set_title("The lasso: fitting and validation RMSE for each penalty")
ax.legend()
plt.show()

The two columns tell different stories. The fitting RMSE rises with the
penalty, as in Part 2. The validation RMSE first **falls**, from 30.26 to its
lowest value, 29.75 at a penalty of 0.3, and then rises again, to 31.85 at 10.

Why? With a small penalty the model follows the noise of the 450 fitting
people; that noise does not repeat in the 150 validation people, so their
predictions are worse (**overfitting**). With a large penalty the model is too
restrained to follow even the real pattern, and both errors are high
(**underfitting**). The validation rows played no part in the fitting, so they
behave like new patients, and their error shows the cost of both. The
penalty 0.3 is chosen because it has the lowest validation RMSE. Choosing a
hyperparameter on rows the model was not fitted on is the normal way to choose
it.

> **Coming later: cross-validation**
>
> One validation set is the simplest way to compare candidates, but the result
> depends on which 150 people landed in it (try the Stretch). In Exercise 5.4
> you will learn **cross-validation**, a resampling method that repeats the
> comparison on several different splits of the training rows and averages.

> **Learning points**
>
> - Candidates are scored on validation rows that were not used to fit them; the test rows are kept for one final check.
> - Too small a penalty overfits, too large a penalty underfits; the validation error is lowest in between.

------------------------------------------------------------------------

### Part 4 — The same search with `GridSearchCV()`

scikit-learn runs this loop for you. `GridSearchCV()` takes the pipeline, the
grid of candidates, and the split to score them on. `PredefinedSplit()` gives
it our one split: rows marked `-1` are always used for fitting, rows marked `0`
are the validation rows. `GridSearchCV()` always looks for the **highest**
score, so for an error it uses the negative RMSE.

In [ ]:
one_split = PredefinedSplit(np.where(is_val, 0, -1))   # 0 = validation row, -1 = fitting row

lasso_search = GridSearchCV(lasso_pipe,
                            param_grid={"model__alpha": penalties},
                            cv=one_split,
                            scoring="neg_root_mean_squared_error")
lasso_search.fit(train[predictors], train["glucose"])

results = pd.DataFrame(lasso_search.cv_results_)
results["val_rmse"] = -results["mean_test_score"]      # back to a positive RMSE
print(results[["param_model__alpha", "val_rmse"]].round(2))
print(lasso_search.best_params_)

The `val_rmse` column is the validation RMSE of your loop, number for number,
and `best_params_` is the same penalty, 0.3. (scikit-learn calls the held-out
rows "test" in its names, such as `mean_test_score`; here they are our
validation rows.) `ValidationCurveDisplay` draws the fitting and the validation
error for each value:

In [ ]:
display = ValidationCurveDisplay.from_estimator(
    lasso_pipe, train[predictors], train["glucose"],
    param_name="model__alpha", param_range=penalties,
    cv=one_split, scoring="neg_root_mean_squared_error", negate_score=True,
    std_display_style=None)
display.ax_.set_xscale("log")
display.ax_.legend(["fitting rows", "validation rows"])
display.ax_.set_title("ValidationCurveDisplay: RMSE for each penalty")
plt.show()

| Your loop in Part 3 | `GridSearchCV()` |
|:--------------------|:-----------------|
| `for p in penalties:` | `param_grid={"model__alpha": penalties}` |
| `fit(fitting[...])` | the rows marked `-1` in `one_split` |
| score on `validation` with `root_mean_squared_error` | the rows marked `0`, `scoring="neg_root_mean_squared_error"` |
| read the table for the lowest `val_rmse` | `best_params_` |

**Which hyperparameters does a model have?** `get_params()` lists every
setting of every step of a pipeline, under the names `set_params()` and
`param_grid` use; the
[scikit-learn page for `ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)
explains `alpha` and `l1_ratio`. The user guide's
[tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
covers the search tools.

In [ ]:
[name for name in lasso_pipe.get_params() if name.startswith("model__")]

> **Learning points**
>
> - `GridSearchCV()` fits and scores every candidate; `cv_results_`, `best_params_` and `ValidationCurveDisplay` show and choose.
> - It is the loop of Part 3, written for you.

------------------------------------------------------------------------

### Part 5 — Tuning the mix of lasso and ridge

`ElasticNet()` has a second hyperparameter, `l1_ratio`: 1 is the lasso, 0 is
ridge, and values in between mix the two penalties. Keep the penalty at 0.3
and tune only `l1_ratio`:

> **Stuck?**
>
> `l1_ratio` runs from 0 (ridge) to 1 (lasso).

> **Worked answer**

In [ ]:
mix_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("model", ElasticNet(alpha=0.3, max_iter=100000)),
])

mix_search = GridSearchCV(mix_pipe,
                          param_grid={"model__l1_ratio": [0, 0.25, 0.5, 0.75, 1]},
                          cv=one_split,
                          scoring="neg_root_mean_squared_error")
mix_search.fit(train[predictors], train["glucose"])

mix_results = pd.DataFrame(mix_search.cv_results_)
mix_results["val_rmse"] = -mix_results["mean_test_score"]
print(mix_results[["param_model__l1_ratio", "val_rmse"]].round(2))
print(mix_search.best_params_)

> The lasso (`l1_ratio = 1`) is best, at 29.75, against 30.02 to 30.08 for the
> mixes and ridge. The differences are small.

> **Learning points**
>
> - Tune one hyperparameter at a time by fixing the others in the pipeline and putting only one in `param_grid`.

------------------------------------------------------------------------

### Part 6 — Refitting the chosen model, and its one test

During the search every candidate was fitted on the 450 fitting rows only.
Once the best hyperparameters are known, that one model is fitted again on all
600 training rows (fitting and validation together): the same settings, more
data. This is the **refit**. Do it by hand first, using the results table,
then with the tool, and check that the two agree.

**By hand.** Take the row with the lowest validation RMSE, put its value into a
fresh copy of the pipeline with `set_params()`, and fit. `clone()` makes an
unfitted copy of the pipeline, so the one inside the search is left alone.

> **Stuck?**
>
> The refit uses every training row, fitting and validation together.

> **Worked answer**

In [ ]:
from sklearn.base import clone

best = mix_results.loc[mix_results["val_rmse"].idxmin(), "param_model__l1_ratio"]   # the lowest validation RMSE
print("best l1_ratio:", best)

refit_by_hand = clone(mix_pipe).set_params(model__l1_ratio=best)   # put the chosen value in
refit_by_hand.fit(train[predictors], train["glucose"])             # all 600 training rows

**With the tool.** `GridSearchCV()` has already done the refit: after the
search it puts the best value into the pipeline and fits it on all the rows it
was given (`refit=True`, the default). The result is in `best_estimator_`.

In [ ]:
refit_by_tool = mix_search.best_estimator_

Are they the same model? Put the coefficients side by side:

In [ ]:
coefs = pd.DataFrame({"by_hand": refit_by_hand["model"].coef_,
                      "best_estimator_": refit_by_tool["model"].coef_},
                     index=refit_by_tool[:-1].get_feature_names_out())   # the 35 column names
print(coefs.head().round(3))
print("same model:", np.allclose(coefs["by_hand"], coefs["best_estimator_"]))

`True`: every coefficient is the same. `best_estimator_` is the refit you did
by hand.

Now the **one** test check, with the unpenalised model for comparison:

In [ ]:
plain_pipe = Pipeline(mix_pipe.steps[:-1] + [("model", LinearRegression())])
plain_pipe.fit(train[predictors], train["glucose"])

print("chosen model, test RMSE:     ", round(root_mean_squared_error(test["glucose"], refit_by_tool.predict(test[predictors])), 2))
print("unpenalised model, test RMSE:", round(root_mean_squared_error(test["glucose"], plain_pipe.predict(test[predictors])), 2))
print("columns kept:", (refit_by_tool["model"].coef_ != 0).sum())

The chosen model (penalty 0.3, `l1_ratio` 1, the lasso) has a test RMSE of
27.04 against 27.12 for the unpenalised model, and it uses 14 of the 35
columns.

**Question!** The penalised model is only a little more accurate on the test
rows, but it uses far fewer columns. Why might that still matter?

> **From one validation set to cross-validation**
>
> In Exercise 5.4 the only change is `cv`: instead of our one split, `cv=5` makes
> five, and `GridSearchCV()` averages over them. `best_params_` and
> `best_estimator_` work as here.
>
> ```python
> GridSearchCV(lasso_pipe, param_grid=..., cv=one_split, ...)   # this exercise
> GridSearchCV(lasso_pipe, param_grid=..., cv=5, ...)           # Exercise 5.4
> ```

> **Learning points**
>
> - Once the hyperparameters are chosen, refit that one model on all the training rows: by hand with `clone()`, `set_params()` and `fit()`, or let `GridSearchCV()` do it and take `best_estimator_`. Both give the same model.
> - Then check it once on the test rows.

> **Other ways to choose the candidates**
>
> A regular grid is the simplest way to choose which values to try, but not the
> only one. **Random search** draws the candidates at random from a range
> (`grid_random()` in tidymodels, `RandomizedSearchCV()` in scikit-learn).
> **Space-filling designs**, such as the **Latin hypercube**, spread a fixed
> number of candidates evenly over the range
> ([`grid_space_filling()`](https://dials.tidymodels.org/reference/grid_space_filling.html)).
> **Iterative** searches, such as Bayesian optimisation, choose each new
> candidate from the results so far. All of them score the candidates the same
> way as here. Read more in chapters 13 and 14 of Tidy Modeling with R
> ([grid search](https://www.tmwr.org/grid-search),
> [iterative search](https://www.tmwr.org/iterative-search)) and in the
> scikit-learn guide to
> [tuning hyper-parameters](https://scikit-learn.org/stable/modules/grid_search.html).

## Stretch

> **Stretch — a different validation set**
>
> Make the validation rows every fourth training row starting from the second
> (`seq(2, nrow(train), by = 4)` in R, `np.arange(len(train)) % 4 == 1` in
> Python) and run the search of Part 3 or Part 4 again. Is the same penalty
> chosen? What does that tell you about choosing with a single validation set?